In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential, Model
from keras.layers import LSTM, Dense, Dropout, Input, RepeatVector
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, classification_report, matthews_corrcoef
import matplotlib.pyplot as plt
import scikitplot as skplt
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import LSTM, GRU
from keras.layers import Dense , Dropout
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.multiclass import OneVsRestClassifier
from itertools import cycle
import matplotlib.pyplot as plt
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import scikitplot as skplt
from sklearn.preprocessing import label_binarize
from sklearn.datasets import load_digits
#from yellowbrick.features import ParallelCoordinates
from keras.layers import Bidirectional, LSTM
from keras.layers import Bidirectional, GRU
from sklearn.metrics import matthews_corrcoef

In [ ]:
filepath= "C:/Users/Theda/Desktop/Edge-iiotset.csv"

In [ ]:
mydata= pd.read_csv(filepath)

In [ ]:
mydata.info()

In [ ]:
drop_columns = ["frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4","arp.dst.proto_ipv4", 

         "http.file_data","http.request.full_uri","icmp.transmit_timestamp",

         "http.request.uri.query", "tcp.options","tcp.payload","tcp.srcport",

         "tcp.dstport", "udp.port", "mqtt.msg"]

mydata.drop(drop_columns, axis=1, inplace=True)


In [ ]:
print(mydata['Attack_type'].value_counts())

In [ ]:
mydata.drop('Attack_label', axis='columns', inplace=True)


In [ ]:
mydata.replace([np.inf, -np.inf], np.nan, inplace=True)
mydata.dropna(inplace=True)
mydata.drop_duplicates(inplace=True)
mydata.reset_index(inplace=True, drop=True)
mydata = mydata.sample(frac=1).reset_index(drop=True)

In [ ]:
mydata.info()

In [ ]:
attacks = {'Normal': 0 ,'DDoS_UDP' :1, 'DDoS_ICMP':2, 'SQL_injection':3, 'Password':4,
       'Vulnerability_scanner':5 , 'DDoS_TCP':6, 'DDoS_HTTP':7, 'Uploading':8, 'Backdoor':9, 
       'Port_Scanning':10, 'XSS':11, 'Ransomware':12, 'Fingerprinting':13, 'MITM':14}
mydata['label']=mydata['Attack_type'].map(attacks)

In [ ]:
print(mydata['label'].value_counts())

In [ ]:
mydata.drop('Attack_type', axis='columns', inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()

In [ ]:
mydata['mqtt.protoname'] = mydata['mqtt.protoname'].astype(str)
mydata['mqtt.topic'] = mydata['mqtt.topic'].astype(str)
mydata['mqtt.conack.flags'] = mydata['mqtt.conack.flags'].astype(str)
mydata['dns.qry.name.len'] = mydata['dns.qry.name.len'].astype(str)
mydata['http.request.method'] = mydata['http.request.method'].astype(str)
mydata['http.referer'] = mydata['http.referer'].astype(str)
mydata['http.request.version'] = mydata['http.request.version'].astype(str)

In [ ]:
from sklearn.preprocessing import LabelEncoder

labelencoder = LabelEncoder()

mydata['mqtt.protoname'] = labelencoder.fit_transform(mydata['mqtt.protoname'])
mydata['mqtt.topic'] = labelencoder.fit_transform(mydata['mqtt.topic'])
mydata['mqtt.conack.flags'] = labelencoder.fit_transform(mydata['mqtt.conack.flags'])
mydata['dns.qry.name.len'] = labelencoder.fit_transform(mydata['dns.qry.name.len'])
mydata['http.request.method'] = labelencoder.fit_transform(mydata['http.request.method'])
mydata['http.referer'] = labelencoder.fit_transform(mydata['http.referer'])
mydata['http.request.version'] = labelencoder.fit_transform(mydata['http.request.version'])


In [ ]:
mydata.info()

In [ ]:
X = mydata.iloc[:, 0:46]
Y = mydata.iloc[:, 46:47]

In [ ]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X = sc.fit_transform(X)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.30, random_state=42)
x_train = np.array(x_train)
x_train = np.reshape (x_train,(x_train.shape[0], 1, x_train.shape[1]))
#Reshape and normalize test data
x_test = np.array(x_test)
x_test = np.reshape (x_test,(x_test.shape[0], 1, x_test.shape[1]))

In [ ]:
Num_classes = len(np.unique(Y))
y_train_ohe=to_categorical(y_train,Num_classes)
y_train_ohe = pd.DataFrame(y_train_ohe)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
import numpy as np
import tensorflow as tf

In [ ]:
def sampling(args):
    z_mean, z_log_var = args
    batch = K.shape(z_mean)[0]
    dim = K.int_shape(z_mean)[1]
    epsilon = K.random_normal(shape=(batch, dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

In [ ]:
beta = 4.0  # (beta >= 1)

In [ ]:
input_dim = x_train.shape[2]  # This should be 8, representing data[0] to data[7]
inputs = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(inputs)
z_mean = Dense(32)(encoded)
z_log_var = Dense(32)(encoded)

In [ ]:
z = Lambda(sampling, output_shape=(32,))([z_mean, z_log_var])

In [ ]:
decoded = Dense(64, activation='relu')(z)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

In [ ]:
vae = Model(inputs, decoded)

In [ ]:
class KLLossLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(KLLossLayer, self).__init__(**kwargs)
    
    def call(self, inputs):
        z_mean, z_log_var = inputs
        kl_loss = -0.5 * K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
        self.add_loss(beta * K.mean(kl_loss))  # Add the KL loss to the final model loss
        return z_mean


In [ ]:
z_mean_kl_loss = KLLossLayer()([z_mean, z_log_var])

In [ ]:
def beta_vae_loss(inputs, decoded):
    # Reconstruction loss (Mean Squared Error)
    reconstruction_loss = K.mean(K.square(inputs - decoded), axis=-1)
    return reconstruction_loss  # KL loss is added automatically

In [ ]:
vae.compile(optimizer='adam', loss=beta_vae_loss)

In [ ]:
# Reshape x_train to match the expected input shape (batch_size, input_dim)
x_train_flattened = x_train.reshape((x_train.shape[0], x_train.shape[2]))

In [ ]:
# Train the Beta-VAE
betahistory=vae.fit(x_train_flattened, x_train_flattened, epochs=10, batch_size=128, validation_split=0.5)

In [ ]:
# Use the encoder part for feature extraction
encoder = Model(inputs, z_mean)  # Use the mean as the encoded representation
x_train_encoded = encoder.predict(x_train_flattened)
x_test_encoded = encoder.predict(x_test.reshape((x_test.shape[0], x_test.shape[2])))

x_train_encoded = np.reshape(x_train_encoded, (x_train_encoded.shape[0], 1, x_train_encoded.shape[1]))
x_test_encoded = np.reshape(x_test_encoded, (x_test_encoded.shape[0], 1, x_test_encoded.shape[1]))


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Attention, Input, Dense, LSTM, Bidirectional, Conv1D, LayerNormalization, Add, Dropout, Concatenate, Flatten
from tensorflow.keras.models import Model

# Enhanced Multi-Scale Multi-Head Attention Layer
class MultiScaleMultiHeadAttention(Layer):
    def __init__(self, num_heads=4, scales=[1, 2, 4], feature_dim=None, **kwargs):
        super(MultiScaleMultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.scales = scales
        self.conv_layers = [
            [Conv1D(filters=feature_dim, kernel_size=scale, padding='same', activation='relu')
             for _ in range(num_heads)] for scale in scales]
        self.attention_heads = [[Attention() for _ in range(num_heads)] for _ in self.scales]
        self.fusion_layer = Dense(feature_dim * len(scales) * num_heads, activation='sigmoid')

    def call(self, x):
        multi_scale_outputs = []
        for scale_idx, (scale_heads, scale_conv_layers) in enumerate(zip(self.attention_heads, self.conv_layers)):
            for head_idx, (attention, conv) in enumerate(zip(scale_heads, scale_conv_layers)):
                conv_x = conv(x)
                attention_output = attention([conv_x, conv_x])
                multi_scale_outputs.append(attention_output)
        
        concatenated = Concatenate(axis=-1)(multi_scale_outputs)
        fusion_weight = self.fusion_layer(concatenated)
        fused_output = concatenated * fusion_weight
        return fused_output
        
# Residual BiLSTM block with matching dimensions
def residual_bilstm_block(x, units):
    x_res = x  
    x = Bidirectional(LSTM(units, activation='relu', return_sequences=True))(x)
    x = LayerNormalization()(x)
    x_res = Dense(units * 2)(x_res)  
    x = Add()([x, x_res])  
    return x

num_features = x_train_encoded.shape[2]

inputs = Input(shape=(x_train_encoded.shape[1], x_train_encoded.shape[2]))
x = Bidirectional(LSTM(80, activation='relu', return_sequences=True))(inputs)

x = residual_bilstm_block(x, 32)
x = residual_bilstm_block(x, 16)

x = Dropout(0.2)(x)

# ADD NAME HERE
x = MultiScaleMultiHeadAttention(
        num_heads=4,
        scales=[1, 2, 4],
        feature_dim=num_features,
        name="msmha"     # <<< THIS IS THE IMPORTANT UPDATE
)(x)

x = Flatten()(x)
x = Dropout(0.2)(x)
outputs = Dense(15, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-4),
    metrics=['accuracy']
)

lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, verbose=1
)


In [ ]:
history = model.fit(x_train_encoded, y_train_ohe, epochs=2, validation_split=0.5, batch_size=256)

In [ ]:
#attention interpretation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Your attack mapping
attacks = {
    'Normal': 0,
    'DDoS_UDP': 1,
    'DDoS_ICMP': 2,
    'SQL_injection': 3,
    'Password': 4,
    'Vulnerability_scanner': 5,
    'DDoS_TCP': 6,
    'DDoS_HTTP': 7,
    'Uploading': 8,
    'Backdoor': 9,
    'Port_Scanning': 10,
    'XSS': 11,
    'Ransomware': 12,
    'Fingerprinting': 13,
    'MITM': 14
}

num_classes = len(attacks)

# index → name list
attack_names = [None] * num_classes
for name, idx in attacks.items():
    attack_names[idx] = name

print("Attack index to name mapping:", attack_names)


In [ ]:
# Convert y_test to a clean 1D numpy array of ints
try:
    import pandas as pd
    if isinstance(y_test, (pd.Series, pd.DataFrame)):
        true_classes = y_test.values.ravel()
    else:
        true_classes = np.asarray(y_test).ravel()
except ImportError:
    true_classes = np.asarray(y_test).ravel()

true_classes = true_classes.astype(int)

print("true_classes shape:", true_classes.shape)
print("Unique labels in test:", np.unique(true_classes))


In [ ]:
# Build a sub-model that outputs the MSMHA fused representation
att_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("msmha").output
)

# Check input shapes
T = x_train_encoded.shape[1]   # "sequence" dimension (can be 1)
F = x_train_encoded.shape[2]   # features per sample

num_scales = 3      # [1, 2, 4]
num_heads = 4

print(f"T = {T}, F = {F}, num_scales = {num_scales}, num_heads = {num_heads}")


In [ ]:
# Predictions on test set
pred_probs = model.predict(x_test_encoded, verbose=0)
pred_classes = np.argmax(pred_probs, axis=1)

print("pred_classes shape:", pred_classes.shape)

# Indices of correctly classified samples
correct_idx = np.where(pred_classes == true_classes)[0]
print(f"Correctly classified test samples: {len(correct_idx)}")

# Use up to N samples for interpretation (for speed)
N_SAMPLES = min(2000, len(correct_idx))
selected_idx = correct_idx[:N_SAMPLES]

x_eval = x_test_encoded[selected_idx]
y_eval_true = true_classes[selected_idx]
y_eval_pred = pred_classes[selected_idx]

print(f"Using {N_SAMPLES} correctly classified samples for MSMHA analysis.")


In [ ]:
# Get MSMHA outputs for the selected samples
att_outputs = att_model.predict(x_eval, verbose=0)  # shape: (N, T, fused_dim)
print("att_outputs shape:", att_outputs.shape)

N = att_outputs.shape[0]
fused_dim = att_outputs.shape[-1]
expected_dim = num_scales * num_heads * F

print("fused_dim:", fused_dim, " expected_dim:", expected_dim)
assert fused_dim == expected_dim, f"Expected fused_dim={expected_dim}, got {fused_dim}"

# Average over T (if T>1); if T==1 this just squeezes that dimension
att_timeavg = att_outputs.mean(axis=1)   # shape: (N, fused_dim)
print("att_timeavg shape:", att_timeavg.shape)

# Reshape into (N, S, H, F)
att_shaped = att_timeavg.reshape(
    N,
    num_scales,
    num_heads,
    F
)
print("att_shaped shape (N, S, H, F):", att_shaped.shape)


In [ ]:
# Initialize accumulators
class_scale_raw = np.zeros((num_classes, num_scales))            # (C, S)
class_head_raw = np.zeros((num_classes, num_scales, num_heads))  # (C, S, H)
class_feature_raw = np.zeros((num_classes, F))                   # (C, F)
class_counts = np.zeros(num_classes, dtype=int)

# Aggregate over samples
for i in range(N):
    c = y_eval_true[i]
    if c < 0 or c >= num_classes:
        continue  # ignore invalid labels

    class_counts[c] += 1

    H_sample = att_shaped[i]   # shape: (S, H, F)

    # Scale importance: mean over heads and features -> (S,)
    class_scale_raw[c] += H_sample.mean(axis=(1, 2))

    # Head importance: mean over features -> (S, H)
    class_head_raw[c] += H_sample.mean(axis=2)

    # Feature importance: mean over scales and heads -> (F,)
    class_feature_raw[c] += H_sample.mean(axis=(0, 1))

print("Class sample counts (correctly classified):", class_counts)


In [ ]:
eps = 1e-8

# Normalize by sample counts
for c in range(num_classes):
    if class_counts[c] > 0:
        class_scale_raw[c] /= class_counts[c]
        class_head_raw[c] /= class_counts[c]
        class_feature_raw[c] /= class_counts[c]

# 1) Scale-wise: per class, sum over scales = 1
class_scale_importance = class_scale_raw / (class_scale_raw.sum(axis=1, keepdims=True) + eps)

# 2) Head-wise: per class, sum over all (S,H) = 1
class_head_importance = class_head_raw / (class_head_raw.sum(axis=(1, 2), keepdims=True) + eps)

# 3) Feature-wise: per class, sum over features = 1
class_feature_importance = class_feature_raw / (class_feature_raw.sum(axis=1, keepdims=True) + eps)

print("Scale importance shape:", class_scale_importance.shape)     # (C, S)
print("Head importance shape:", class_head_importance.shape)       # (C, S, H)
print("Feature importance shape:", class_feature_importance.shape) # (C, F)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def inspect_class_attention(class_name_or_id, top_k_features=15):
    """
    Strong interpretability for one attack class:
      - normalized scale-wise importance
      - normalized head-wise importance
      - top-K most important features (by index)

    class_name_or_id: either int (0–14) or string key from `attacks`
    """

    # ---------- Resolve class id ----------
    if isinstance(class_name_or_id, str):
        if class_name_or_id not in attacks:
            raise ValueError(f"Unknown class name: {class_name_or_id}")
        cid = attacks[class_name_or_id]
    else:
        cid = int(class_name_or_id)

    if cid < 0 or cid >= len(attack_names):
        raise ValueError(f"Class id {cid} out of range.")

    name = attack_names[cid]
    count = class_counts[cid]

    print(f"\n===== ATTENTION INTERPRETATION FOR CLASS {cid} ({name}) =====")
    print(f"Correctly classified samples used for this class: {count}")
    if count == 0:
        print("No correctly classified samples for this class → cannot interpret.")
        return

    scales = [1, 2, 4]

    # ---------- Scale-wise ----------
    print("\nScale-wise normalized importance:")
    for s_idx, s in enumerate(scales):
        val = float(class_scale_importance[cid, s_idx])
        print(f"  Scale = {s}: {val:.6f}")

    plt.figure(figsize=(4, 3))
    plt.bar(scales, class_scale_importance[cid])
    plt.xlabel("Temporal scale (Conv1D kernel size)")
    plt.ylabel("Normalized importance")
    plt.title(f"Scale-wise importance for {name}")
    plt.tight_layout()
    plt.show()

    # ---------- Head-wise ----------
    print("\nHead-wise normalized importance (per scale):")
    labels = []
    values = []
    for s_idx, s in enumerate(scales):
        for h in range(num_heads):
            val = float(class_head_importance[cid, s_idx, h])
            print(f"  Scale={s}, Head={h+1}: {val:.6f}")
            labels.append(f"S{s}-H{h+1}")
            values.append(val)

    plt.figure(figsize=(6, 3))
    plt.bar(np.arange(len(labels)), values)
    plt.xticks(np.arange(len(labels)), labels, rotation=45)
    plt.ylabel("Normalized importance")
    plt.title(f"Head-wise importance for {name}")
    plt.tight_layout()
    plt.show()

    # ---------- Feature-wise (top-K) ----------
    feat_scores = class_feature_importance[cid]   # shape: (F,)
    feat_ranking = list(enumerate(feat_scores))
    feat_ranking.sort(key=lambda x: x[1], reverse=True)

    print(f"\nTop {top_k_features} most important features for {name}:")
    for idx, score in feat_ranking[:top_k_features]:
        print(f"  Feature {idx}: {float(score):.6f}")

    top_idx = [idx for idx, _ in feat_ranking[:top_k_features]]
    top_vals = [score for _, score in feat_ranking[:top_k_features]]

    plt.figure(figsize=(6, 3))
    plt.bar(np.arange(len(top_idx)), top_vals)
    plt.xticks(np.arange(len(top_idx)), [str(i) for i in top_idx], rotation=45)
    plt.xlabel("Feature index")
    plt.ylabel("Normalized importance")
    plt.title(f"Top-{top_k_features} feature importance for {name}")
    plt.tight_layout()
    plt.show()


# 🔥 CALL IT HERE SO YOU SEE SOMETHING IMMEDIATELY
inspect_class_attention("Normal", top_k_features=15)
# you can change "Normal" to "DDoS_UDP", "Ransomware", "MITM", etc.


In [ ]:
# If you’re in Jupyter / Spyder, also make sure:
# %matplotlib inline   # (only needed in notebooks, can skip if you see other plots)

inspect_class_attention("Normal", top_k_features=15)
inspect_class_attention("DDoS_UDP", top_k_features=15)
inspect_class_attention("Ransomware", top_k_features=15)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def inspect_class_attention_line(class_name_or_id, top_k_features=20):
    """
    High-quality line-graph interpretation for one attack class:
      - Scale-wise importance (line graph)
      - Head-wise importance (line graph)
      - Feature importance (top K, line graph)
    """

    print(">>> Function called with:", class_name_or_id)  # DEBUG LINE

    # ---------- Resolve class id ----------
    if isinstance(class_name_or_id, str):
        if class_name_or_id not in attacks:
            raise ValueError(f"Unknown class: {class_name_or_id}")
        cid = attacks[class_name_or_id]
    else:
        cid = int(class_name_or_id)

    if cid < 0 or cid >= len(attack_names):
        raise ValueError(f"Class id {cid} out of range.")

    name = attack_names[cid]
    count = class_counts[cid]

    print(f"\n===== LINE-GRAPH ATTENTION INTERPRETATION FOR CLASS {cid} ({name}) =====")
    print(f"Correctly classified samples: {count}")
    if count == 0:
        print("No correctly classified samples for this class → cannot interpret.")
        return

    scales = [1, 2, 4]

    # ----------------------------------------------------------
    # 1. SCALE-WISE IMPORTANCE (LINE GRAPH)
    # ----------------------------------------------------------
    print("\nScale-wise importance:")
    for s_idx, s in enumerate(scales):
        print(f"  Scale {s}: {class_scale_importance[cid, s_idx]:.6f}")

    plt.figure(figsize=(5, 3))
    plt.plot(scales, class_scale_importance[cid], marker='o', linewidth=2)
    plt.xlabel("Temporal scale (Conv1D kernel size)")
    plt.ylabel("Normalized importance")
    plt.title(f"Scale-wise attention trend for {name}")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # ----------------------------------------------------------
    # 2. HEAD-WISE IMPORTANCE (LINE GRAPH)
    # ----------------------------------------------------------
    head_flat = class_head_importance[cid].reshape(-1)  # (S*H,)
    head_indices = np.arange(len(head_flat))

    print("\nHead-wise importance (flattened across scales):")
    for idx, score in enumerate(head_flat):
        s = scales[idx // num_heads]
        h = (idx % num_heads) + 1
        print(f"  S{s}-H{h}: {score:.6f}")

    plt.figure(figsize=(8, 3))
    plt.plot(head_indices, head_flat, marker='o', linewidth=2)
    plt.xticks(
        head_indices,
        [f"S{scales[i//num_heads]}-H{(i%num_heads)+1}" for i in head_indices],
        rotation=45
    )
    plt.ylabel("Normalized importance")
    plt.title(f"Head-wise attention trend for {name}")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # ----------------------------------------------------------
    # 3. FEATURE IMPORTANCE (TOP-K LINE GRAPH)
    # ----------------------------------------------------------
    feat_scores = class_feature_importance[cid]
    feat_rank = list(enumerate(feat_scores))
    feat_rank.sort(key=lambda x: x[1], reverse=True)

    top_idx = [idx for idx, _ in feat_rank[:top_k_features]]
    top_vals = [score for _, score in feat_rank[:top_k_features]]

    print(f"\nTop {top_k_features} features for {name}:")
    for idx, score in zip(top_idx, top_vals):
        print(f"  Feature {idx}: {score:.6f}")

    plt.figure(figsize=(7, 3))
    plt.plot(top_idx, top_vals, marker='o', linewidth=2)
    plt.xlabel("Feature index")
    plt.ylabel("Normalized importance")
    plt.title(f"Top-{top_k_features} feature importance trend for {name}")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


# 🔥 ACTUALLY CALL IT HERE
inspect_class_attention_line("Normal", top_k_features=20)
# change "Normal" to any other label e.g. "DDoS_UDP", "Ransomware", "MITM"


In [ ]:
# Classes to include in the multi-class plots (edit as you like)
classes_to_plot = ["DDoS_UDP", "SQL_injection", "Uploading", "Backdoor", "Port_Scanning", "XSS", "Ransomware", "Fingerprinting", "MITM"]

# Map to indices and filter out classes with zero correctly classified samples
class_ids = []
class_labels = []
for cname in classes_to_plot:
    cid = attacks[cname]
    if class_counts[cid] > 0:
        class_ids.append(cid)
        class_labels.append(cname)
    else:
        print(f"Skipping {cname}: no correctly classified samples.")

print("Classes used in multi-class plots:", class_labels)


In [ ]:
# Build x-axis indices and labels for all heads across all scales
head_indices = np.arange(num_scales * num_heads)
scale_values = [1, 2, 4]

head_xticks = []
for i in head_indices:
    s = scale_values[i // num_heads]
    h = (i % num_heads) + 1
    head_xticks.append(f"S{s}-H{h}")

plt.figure(figsize=(5.5, 4))

for cid, cname in zip(class_ids, class_labels):
    # class_head_importance[cid] -> shape (S, H)
    head_flat = class_head_importance[cid].reshape(-1)  # (S*H,)
    plt.plot(head_indices, head_flat, marker='o', linewidth=2, label=cname)

plt.xticks(head_indices, head_xticks, rotation=45)
plt.ylabel("Normalized importance", fontsize=13)
plt.title("Head-wise Attention", fontsize=13)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(
    loc='upper center',
    bbox_to_anchor=(0.44, -0.20),
    ncol=4,
    fontsize=8,
    frameon=True
)
plt.tight_layout()
plt.savefig('HEAD.png', format='png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

scales = [1, 2, 4]
num_scales = len(scales)
num_classes_to_plot = len(class_ids)

bar_width = 0.8 / num_classes_to_plot
x = np.arange(num_scales)

# Strong, high-contrast, publication-grade palette
color_map = get_cmap('Dark2')   # <<< BEST CHOICE FOR PAPERS

plt.figure(figsize=(4.5, 4), dpi=220)

for i, (cid, cname) in enumerate(zip(class_ids, class_labels)):
    y = class_scale_importance[cid]
    
    plt.bar(
        x + i * bar_width,
        y,
        width=bar_width,
        label=cname,
        color=color_map(i % color_map.N)
    )

plt.xticks(x + bar_width * (num_classes_to_plot-1) / 2, scales)
plt.xlabel("Temporal Scale", fontsize=11)
plt.ylabel("Normalized Importance", fontsize=11)
plt.title("Scale-Wise Attention", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.legend(
    loc='upper center',
    bbox_to_anchor=(0.45, -0.20),
    ncol=4,
    fontsize=6.5,
    frameon=True
)

plt.tight_layout()
plt.savefig('SCAL.png', format='png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# Build x-axis indices and labels for all heads across all scales
head_indices = np.arange(num_scales * num_heads)   # e.g., 12 positions if 3 scales * 4 heads
scale_values = [1, 2, 4]

head_xticks = []
for i in head_indices:
    s = scale_values[i // num_heads]
    h = (i % num_heads) + 1
    head_xticks.append(f"S{s}-H{h}")

num_classes_to_plot = len(class_ids)
bar_width = 0.8 / num_classes_to_plot   # group width = 0.8

color_map = get_cmap('tab20')

plt.figure(figsize=(6, 5))

for k, (cid, cname) in enumerate(zip(class_ids, class_labels)):
    # class_head_importance[cid] has shape (num_scales, num_heads)
    head_flat = class_head_importance[cid].reshape(-1)  # shape (S*H,)

    # Shift each class’s bars slightly so they form groups
    plt.bar(
        head_indices + k * bar_width,
        head_flat,
        width=bar_width,
        label=cname,
        color=color_map(k)
    )

plt.xticks(
    head_indices + bar_width * (num_classes_to_plot - 1) / 2,
    head_xticks,
    rotation=45
)
plt.ylabel("Normalized importance", fontsize=14)
plt.title("Head-wise Attention", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=4, fontsize=8)

plt.tight_layout()
plt.savefig('HEAD1.png', format='png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

# Build signature matrix
signatures = []
sig_labels = []

for name, cid in attacks.items():
    if class_counts[cid] == 0:
        print(f"Skipping {name}: 0 samples")
        continue

    scale_sig = class_scale_importance[cid]
    head_sig  = class_head_importance[cid].reshape(-1)
    sig_vec   = np.concatenate([scale_sig, head_sig])

    signatures.append(sig_vec)
    sig_labels.append(name)

signatures = np.stack(signatures, axis=0)

# PCA
pca = PCA(n_components=2, random_state=0)
sig_2d = pca.fit_transform(signatures)

plt.figure(figsize=(5, 4), dpi=200)

colors = plt.cm.tab20(np.linspace(0, 1, len(sig_labels)))

for (x, y), lbl, c in zip(sig_2d, sig_labels, colors):
    plt.scatter(x, y, color=c, s=70, edgecolors='black', linewidth=0.5, label=lbl)

# One legend entry per class (avoid duplicates)
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(
    by_label.values(),
    by_label.keys(),
    loc='upper center',
    bbox_to_anchor=(0.45, -0.17),
    ncol=4,                 # <-- multiple columns
    fontsize=6,
    frameon=True
)

plt.xlabel("PC1 (Attention Signature)", fontsize=11)
plt.ylabel("PC2 (Attention Signature)", fontsize=11)
plt.title("PCA of Class-wise Attention Signatures", fontsize=11)

plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('PCA.png', format='png', dpi=300, bbox_inches='tight')
plt.show()
